In [1]:
!pip install transformers torch

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

# Load the tokenizer and the BART model (downloads about 1.6 GB the first time)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Use the GPU if Colab gives us one, otherwise the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


def summarize_text(text):
    # 1. Text -> tokens. BART can read up to 1,024 tokens,
    #    so truncation=True cuts longer text instead of crashing.
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt").to(device)

    # 2. BART writes the summary. num_beams=4 means it tries 4 options
    #    at each step and keeps the best, which gives better summaries.
    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=150,
            min_length=50,
            num_beams=4,
            do_sample=False,
        )

    # 3. Tokens -> text
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def summarize_paragraph():
    user_input = input("Enter a large paragraph: ")
    summary = summarize_text(user_input)
    print("\nSummary:")
    print(summary)


summarize_paragraph()

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Enter a large paragraph: The researchers tested a new battery design that uses sodium instead of lithium. Sodium is cheaper and easier to find than lithium, which could make batteries less expensive. In early tests, the new battery kept 90 percent of its charge after 1,000 cycles, which is close to current lithium batteries. However, it stored about 20 percent less energy for its size, so it may suit home energy storage better than electric cars. The team plans to test larger versions next year.

Summary:
Researchers tested a new battery design that uses sodium instead of lithium. Sodium is cheaper and easier to find than lithium, which could make batteries less expensive. In early tests, the new battery kept 90 percent of its charge after 1,000 cycles. However, it stored about 20 percent less energy for its size.
